In [ ]:
import pandas as pd
import glob
import os


import pickle
from pathlib import Path


In [ ]:
results = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-12-03_viral_disease_control_cohort_creation"

results1 = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-12-02_get_age_for_case_and_raw_control_cohorts" 







In [ ]:

def prepare_case_control_data_py(cases_df: pd.DataFrame,
                                controls_df: pd.DataFrame,
                                col_sex: str = "sex_at_birth",
                                col_ancestry: str = "updated_race",
                                col_age: str = "age") -> pd.DataFrame:
    """
    Prepares case and control data for analysis by combining, standardizing 
    variable types, and performing initial sanity checks.

    Args:
        cases_df: DataFrame containing the 'case' subjects.
        controls_df: DataFrame containing the 'control' subjects.
        col_sex: Name of the column representing sex.
        col_ancestry: Name of the column representing ancestry/race.
        col_age: Name of the column representing age.

    Returns:
        A single combined pandas DataFrame with 'case', 'sex', 'ancestry', 
        and 'age' columns standardized.
    """
    
    # 1) Harmonize column names and add case/control indicator
    # (Similar to dplyr::mutate and adding a fixed column)
    cases_df = cases_df.copy()
    controls_df = controls_df.copy()
    
    # Use 1 for Cases (int/integer)
    cases_df['case'] = 1 
    # Use 0 for Controls (int/integer)
    controls_df['case'] = 0

    # 2) Bind, clean types
    # (Similar to dplyr::bind_rows, using pd.concat)
    df = pd.concat([cases_df, controls_df], ignore_index=True)
    
    # 3) Mutate new columns using the string variables (column mapping)
    # This step renames and casts the specified columns to the desired types.
    df['sex'] = df[col_sex].astype('category') # Factors are equivalent to categories
    df['ancestry'] = df[col_ancestry].astype('category')
    df['age'] = pd.to_numeric(df[col_age], errors='coerce') # Similar to as.numeric

    # Optional: Drop rows where the critical new variables are missing 
    # (The R code didn't explicitly drop missing, but this is often good practice)
    # df = df.dropna(subset=['sex', 'ancestry', 'age', 'case']) 

    # 4) Quick sanity checks (prints to the console)
    print("--- Sanity Checks ---")
    
    # Frequency table for Cases vs. Controls (table(df$case))
    print("\nCases vs. Controls:")
    print(df['case'].value_counts())
    
    # Cross-tabulation for Sex vs. Case (table(df$sex, df$case))
    print("\nSex vs. Case:")
    print(pd.crosstab(df['sex'], df['case']))
    
    # Cross-tabulation for Ancestry vs. Case (table(df$ancestry, df$case))
    print("\nAncestry vs. Case:")
    print(pd.crosstab(df['ancestry'], df['case']))
    
    # Summary for Age (summary(df$age))
    print("\nAge Summary:")
    print(df['age'].describe())
    print("---------------------")

    # 5) Return the combined data frame
    return df



In [ ]:


# Assuming the prepare_case_control_data_py function is defined as above:
# from your_module import prepare_case_control_data_py 

def concat_matched_case_and_controls(case_folder, control_folder):
    """
    Loops through case cohort CSV files, finds the matching control cohort 
    file, and processes them using the preparation function.

    Args:
        case_folder (str): Path to the folder containing case CSV files.
        control_folder (str): Path to the folder containing control CSV files.

    Returns:
        dict: A dictionary where keys are cohort names (e.g., 'acute_bronchiolitis_due_to_respiratory_syncytial_virus_white')
              and values are the combined, processed pandas DataFrames.
    """
    processed_data = {}
    
    # 1. Use glob to find all case files
    case_files = glob.glob(os.path.join(case_folder, "*_case_cohort.csv"))
    
    if not case_files:
        print(f"No case files found in: {case_folder}")
        return processed_data

    print(f"Found {len(case_files)} case cohort files to process.")

    # 2. Loop through each case file
    for case_path in case_files:
        # Extract the base name (the common part of the file name)
        # e.g., 'acute_bronchiolitis_due_to_respiratory_syncytial_virus', 'white'
        base_name = os.path.basename(case_path).replace('_case_cohort.csv', '')
        
        # 3. Construct the corresponding control file name
        # The structure is: base_name + '_controls.csv'
        control_file_name = base_name + '_controls.csv'
        control_path = os.path.join(control_folder, control_file_name)
        
        # 4. Check if the control file exists
        if not os.path.exists(control_path):
            print(f"⚠️ Skipping: Control file not found for {base_name} at {control_path}")
            continue

        # 5. Read the files and process
        print(f"\n✅ Processing Cohort: {base_name}...")
        
        try:
            # Read DataFrames
            cases_df = pd.read_csv(case_path)
            controls_df = pd.read_csv(control_path)
            
            # Use the preparation function
            # NOTE: Assuming prepare_case_control_data_py is defined or imported
            combined_df = prepare_case_control_data_py(
                cases_df, 
                controls_df, 
                col_sex="sex_at_birth", # Use your actual column names
                col_ancestry="updated_race",
                col_age="age"
            )
            
            # Clean up the key name for the dictionary
            # Removes the tuple parentheses and spaces to make a cleaner key
            dict_key = base_name.strip("()").replace("'", "").replace(", ", "_")
            processed_data[dict_key] = combined_df
            
            print(f"   -> Successfully combined and processed {len(combined_df)} records.")

        except Exception as e:
            print(f"❌ Error processing {base_name}: {e}")
            
    return processed_data



In [ ]:
#function call

# --- Example Usage (requires the prepare_case_control_data_py function) ---
# NOTE: Replace 'b1_cases' and 'b1_controls' with the actual paths to your folders.
all_processed_cohorts = process_matched_cohorts(f'{results1}/b1_cases', f'{results1}/b1_controls')
print("\n--- Summary of Results ---")
for name, df in all_processed_cohorts.items():
     print(f"Cohort '{name}': {len(df)} rows")

In [ ]:
all_processed_cohorts